# Clipt — Roboflow Model Training v2

Expanded training notebook covering **all sports** supported by Clipt:
- Basketball (PRIORITY 1 — replace failed 0.10 mAP model)
- Lacrosse (PRIORITY 2 — new sport support)
- Football (PRIORITY 3 — additional data)
- General jersey number detection (PRIORITY 4 — universal fallback)
- Soccer, volleyball (PRIORITY 5 — future expansion)

## Prerequisites
- **Runtime: T4 GPU** (Runtime → Change runtime type → T4 GPU)
- **Roboflow API key** stored in Colab Secrets as `ROBOFLOW_API_KEY`
- Training takes ~4-6 hours total on Colab free tier

## After Training
Download all `.pt` files with mAP50 >= 0.5 and place them in:
```
jersey-detection/app/model/
```
Then commit and push to GitHub — Railway auto-deploys.

## Section 1 — Setup

In [ ]:
import os

# Read API key from Colab Secrets (key icon in sidebar)
from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')

!pip install roboflow ultralytics -q
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key=api_key)

# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Training will be very slow.")

## Section 2 — Download ALL Datasets

Run each cell to download datasets organized by priority.

### PRIORITY 1: Basketball Jersey Number Detection
The previous basketball_jersey_ocr.pt had mAP50: 0.10 — completely unusable.
We need a much better basketball jersey number model.

In [ ]:
# Dataset P1A — VolleyAi Jersey Number Detection (6,932 images)
# BEST overall jersey number dataset — multi-sport, huge, well-annotated
# Classes: jersey number digits
# License: CC BY 4.0
project_p1a = rf.workspace("volleyai-actions").project("jersey-number-detection-s01j4")
dataset_p1a = project_p1a.version(2).download("yolov8")
print(f"P1A location: {dataset_p1a.location}")

In [ ]:
# Dataset P1B — Roboflow Basketball Jersey Numbers OCR v5 (3,615 images)
# NBA Playoffs footage — basketball-specific jersey crops
# This is the replacement for the failed Basketball-Players-1 dataset
# Classes: jersey number OCR
# License: CC BY 4.0
project_p1b = rf.workspace("roboflow-jvuqo").project("basketball-jersey-numbers-ocr")
dataset_p1b = project_p1b.version(5).download("yolov8")
print(f"P1B location: {dataset_p1b.location}")

In [ ]:
# Dataset P1C — Roboflow Basketball Player Detection 2 (1,398 images)
# Player bounding boxes for basketball — use for crop-then-OCR pipeline
# Classes: players
# License: CC BY 4.0
project_p1c = rf.workspace("roboflow-jvuqo").project("basketball-player-detection-2")
dataset_p1c = project_p1c.version(1).download("yolov8")
print(f"P1C location: {dataset_p1c.location}")

### PRIORITY 2: Lacrosse Player + Jersey Detection
Clipt supports lacrosse athletes — we need player detection for this sport.

In [ ]:
# Dataset P2A — RySEAI Lacrosse Object Detection (528 images)
# BEST available lacrosse dataset
# Classes: Goalie, Longpole, Referee, Shortstick, sports ball
# License: CC BY 4.0
project_p2a = rf.workspace("ryseai").project("lacrosse-object-detection")
dataset_p2a = project_p2a.version(1).download("yolov8")
print(f"P2A location: {dataset_p2a.location}")

In [ ]:
# Dataset P2B — Sports Computer Vision / Lacrosse (380 images)
# Additional lacrosse data to supplement P2A
# Classes: Goalie, Lacrosse Ball, Long stick, Referee, Short stick
# License: CC BY 4.0
project_p2b = rf.workspace("computer-vision-ho8xk").project("sports-computer-vision")
dataset_p2b = project_p2b.version(1).download("yolov8")
print(f"P2B location: {dataset_p2b.location}")

### PRIORITY 3: Additional Football Datasets
More training data for American football — different angles and position classification.

In [ ]:
# Dataset P3A — bronkscottema Football Players by Position (755 images)
# American football with position-level classes
# Classes: CENTER, DB, LB, QB, RB, S, SKILL, WR (8 classes)
# License: CC BY 4.0
# v15 has 97.3% mAP, 96.0% precision — excellent quality
project_p3a = rf.workspace("bronkscottema").project("football-players-zm06l")
dataset_p3a = project_p3a.version(15).download("yolov8")
print(f"P3A location: {dataset_p3a.location}")

In [ ]:
# Dataset P3B — Football Presnap Tracker (828 images)
# Presnap formation analysis — useful for play classification
# Classes: football players in formation
# License: CC BY 4.0
project_p3b = rf.workspace("football-tracking").project("football-presnap-tracker")
dataset_p3b = project_p3b.version(1).download("yolov8")
print(f"P3B location: {dataset_p3b.location}")

### PRIORITY 4: General Multi-Sport Jersey Number Detection
Universal fallback layer for any sport.

In [ ]:
# Dataset P4A — Dark Blue JerseyNumbers (826 images)
# Multi-sport jersey number detection
# Classes: number classes
# License: CC BY 4.0
project_p4a = rf.workspace("dark-blue-jt0mg").project("jerseynumbers")
dataset_p4a = project_p4a.version(5).download("yolov8")
print(f"P4A location: {dataset_p4a.location}")

In [ ]:
# Dataset P4B — yakovk Jersey Numbers (556 images)
# General sports jersey number detection
# Classes: player-jersey-numbers
# License: CC BY 4.0
project_p4b = rf.workspace("yakovk").project("jersey-numbers-i1wn5")
dataset_p4b = project_p4b.version(1).download("yolov8")
print(f"P4B location: {dataset_p4b.location}")

### PRIORITY 5: Soccer, Volleyball (future expansion)

In [ ]:
# Dataset P5A — Augmented Startups Soccer Player Detection (1,232 images)
# Soccer player and ball tracking
# Classes: players, ball
# License: CC BY 4.0
project_p5a = rf.workspace("augmented-startups").project("football-player-detection-kucab")
dataset_p5a = project_p5a.version(1).download("yolov8")
print(f"P5A location: {dataset_p5a.location}")

In [ ]:
# Dataset P5B — Volleyball Detection (9,395 images)
# Large volleyball player detection dataset
# Classes: volleyball players
# License: CC BY 4.0
project_p5b = rf.workspace("personal-tajuk").project("volleyball-detection-gs7kt")
dataset_p5b = project_p5b.version(1).download("yolov8")
print(f"P5B location: {dataset_p5b.location}")

## Section 3 — Train ALL Models

Training strategy:
- Jersey number models: epochs=60, augment=True (need precision)
- Player detection models: epochs=50
- Large datasets (1000+): epochs=75
- Small datasets (<500): epochs=100 (more epochs compensates for less data)

In [ ]:
# Model P1A — Basketball/Multi-Sport Jersey Number Detector
# From VolleyAi dataset (6,932 images) — largest jersey number dataset
# This is the PRIMARY replacement for basketball_jersey_ocr.pt
print("=" * 60)
print("TRAINING P1A: Basketball/Multi-Sport Jersey Number Detector")
print(f"Dataset: {dataset_p1a.location}")
print("=" * 60)

model_p1a = YOLO("yolov8n.pt")
model_p1a.train(
    data=f"{dataset_p1a.location}/data.yaml",
    epochs=75,  # Large dataset
    imgsz=640,
    batch=16,
    name="basketball_jersey_number_v2",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)
print("Model P1A training complete!")

In [ ]:
# Model P1B — Basketball Jersey OCR (NBA Playoffs)
# From Roboflow official dataset (3,615 images)
# Basketball-specific OCR — high quality NBA footage
print("=" * 60)
print("TRAINING P1B: Basketball Jersey OCR (NBA Playoffs)")
print(f"Dataset: {dataset_p1b.location}")
print("=" * 60)

model_p1b = YOLO("yolov8n.pt")
model_p1b.train(
    data=f"{dataset_p1b.location}/data.yaml",
    epochs=75,  # Large dataset
    imgsz=640,
    batch=16,
    name="basketball_jersey_ocr_v2",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)
print("Model P1B training complete!")

In [ ]:
# Model P1C — Basketball Player Detector
# For crop-then-OCR pipeline: detect players first, then read jerseys
print("=" * 60)
print("TRAINING P1C: Basketball Player Detector")
print(f"Dataset: {dataset_p1c.location}")
print("=" * 60)

model_p1c = YOLO("yolov8n.pt")
model_p1c.train(
    data=f"{dataset_p1c.location}/data.yaml",
    epochs=75,  # Large dataset
    imgsz=640,
    batch=16,
    name="basketball_player_detector",
    patience=10,
    device=0,
)
print("Model P1C training complete!")

In [ ]:
# Model P2A — Lacrosse Player Detector
# From RySEAI dataset (528 images) — only lacrosse dataset available
# Detects: Goalie, Longpole (defense), Shortstick (offense), Referee
print("=" * 60)
print("TRAINING P2A: Lacrosse Player Detector")
print(f"Dataset: {dataset_p2a.location}")
print("=" * 60)

model_p2a = YOLO("yolov8n.pt")
model_p2a.train(
    data=f"{dataset_p2a.location}/data.yaml",
    epochs=100,  # Small dataset — more epochs
    imgsz=640,
    batch=16,
    name="lacrosse_player_detector",
    patience=20,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print("Model P2A training complete!")

In [ ]:
# Model P3A — Football Position Classifier
# From bronkscottema (755 images) — American football positions
# Classes: CENTER, DB, LB, QB, RB, S, SKILL, WR
print("=" * 60)
print("TRAINING P3A: Football Position Classifier")
print(f"Dataset: {dataset_p3a.location}")
print("=" * 60)

model_p3a = YOLO("yolov8n.pt")
model_p3a.train(
    data=f"{dataset_p3a.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="football_position_classifier",
    patience=15,
    device=0,
)
print("Model P3A training complete!")

In [ ]:
# Model P3B — Football Presnap Detector
# From Football Tracking (828 images) — presnap formation
print("=" * 60)
print("TRAINING P3B: Football Presnap Detector")
print(f"Dataset: {dataset_p3b.location}")
print("=" * 60)

model_p3b = YOLO("yolov8n.pt")
model_p3b.train(
    data=f"{dataset_p3b.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="football_presnap_detector",
    patience=15,
    device=0,
)
print("Model P3B training complete!")

In [ ]:
# Model P4A — Universal Jersey Number Detector
# From Dark Blue (826 images) — multi-sport numbers
print("=" * 60)
print("TRAINING P4A: Universal Jersey Number Detector")
print(f"Dataset: {dataset_p4a.location}")
print("=" * 60)

model_p4a = YOLO("yolov8n.pt")
model_p4a.train(
    data=f"{dataset_p4a.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="universal_jersey_number",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print("Model P4A training complete!")

In [ ]:
# Model P5A — Soccer Player Detector
# From Augmented Startups (1,232 images)
print("=" * 60)
print("TRAINING P5A: Soccer Player Detector")
print(f"Dataset: {dataset_p5a.location}")
print("=" * 60)

model_p5a = YOLO("yolov8n.pt")
model_p5a.train(
    data=f"{dataset_p5a.location}/data.yaml",
    epochs=75,  # Large dataset
    imgsz=640,
    batch=16,
    name="soccer_player_detector",
    patience=10,
    device=0,
)
print("Model P5A training complete!")

In [ ]:
# Model P5B — Volleyball Player Detector
# From Personal (9,395 images) — very large
print("=" * 60)
print("TRAINING P5B: Volleyball Player Detector")
print(f"Dataset: {dataset_p5b.location}")
print("=" * 60)

model_p5b = YOLO("yolov8n.pt")
model_p5b.train(
    data=f"{dataset_p5b.location}/data.yaml",
    epochs=75,  # Large dataset
    imgsz=640,
    batch=16,
    name="volleyball_player_detector",
    patience=10,
    device=0,
)
print("Model P5B training complete!")

## Section 4 — Validate ALL Models & Print mAP50 Scores

Any model with mAP50 < 0.5 is NOT worth deploying.

In [ ]:
print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

models_to_validate = {
    "P1A: Basketball/Multi-Sport Jersey Number v2": "runs/detect/basketball_jersey_number_v2/weights/best.pt",
    "P1B: Basketball Jersey OCR v2 (NBA)": "runs/detect/basketball_jersey_ocr_v2/weights/best.pt",
    "P1C: Basketball Player Detector": "runs/detect/basketball_player_detector/weights/best.pt",
    "P2A: Lacrosse Player Detector": "runs/detect/lacrosse_player_detector/weights/best.pt",
    "P3A: Football Position Classifier": "runs/detect/football_position_classifier/weights/best.pt",
    "P3B: Football Presnap Detector": "runs/detect/football_presnap_detector/weights/best.pt",
    "P4A: Universal Jersey Number": "runs/detect/universal_jersey_number/weights/best.pt",
    "P5A: Soccer Player Detector": "runs/detect/soccer_player_detector/weights/best.pt",
    "P5B: Volleyball Player Detector": "runs/detect/volleyball_player_detector/weights/best.pt",
}

results = {}
deployable = []
skipped = []

for name, path in models_to_validate.items():
    if os.path.exists(path):
        model = YOLO(path)
        metrics = model.val()
        map50 = metrics.box.map50
        map50_95 = metrics.box.map
        results[name] = {"mAP50": map50, "mAP50-95": map50_95, "path": path}
        status = "DEPLOY" if map50 >= 0.5 else "SKIP (< 0.5)"
        if map50 >= 0.5:
            deployable.append((name, path, map50))
        else:
            skipped.append((name, map50))
        print(f"  {name}")
        print(f"    mAP50: {map50:.3f} | mAP50-95: {map50_95:.3f} | {status}")
    else:
        print(f"  {name} — MISSING (training may have failed)")

print("\n" + "=" * 60)
print(f"DEPLOYABLE: {len(deployable)} models (mAP50 >= 0.5)")
for name, _, score in deployable:
    print(f"  ✓ {name} ({score:.3f})")
print(f"\nSKIPPED: {len(skipped)} models (mAP50 < 0.5)")
for name, score in skipped:
    print(f"  ✗ {name} ({score:.3f})")
print("=" * 60)

## Section 5 — Download Only Models with mAP50 >= 0.5

Auto-skips low performers. Download the `.pt` files to your computer.

In [ ]:
from google.colab import files
import shutil

# Map model names to output filenames for the jersey-detection pipeline
output_names = {
    "P1A: Basketball/Multi-Sport Jersey Number v2": "basketball_jersey_number_v2.pt",
    "P1B: Basketball Jersey OCR v2 (NBA)": "basketball_jersey_ocr_v2.pt",
    "P1C: Basketball Player Detector": "basketball_player_detector.pt",
    "P2A: Lacrosse Player Detector": "lacrosse_player_detector.pt",
    "P3A: Football Position Classifier": "football_position_classifier.pt",
    "P3B: Football Presnap Detector": "football_presnap_detector.pt",
    "P4A: Universal Jersey Number": "universal_jersey_number.pt",
    "P5A: Soccer Player Detector": "soccer_player_detector.pt",
    "P5B: Volleyball Player Detector": "volleyball_player_detector.pt",
}

print("Downloading deployable models...\n")
downloaded = 0
for name, path, score in deployable:
    output_name = output_names.get(name, name.replace(" ", "_").lower() + ".pt")
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        shutil.copy(path, output_name)
        files.download(output_name)
        print(f"  ✓ {output_name} ({size_mb:.1f} MB, mAP50: {score:.3f})")
        downloaded += 1

print(f"\n{'=' * 60}")
print(f"DOWNLOADED {downloaded} models")
print(f"\nNext steps:")
print(f"1. Place .pt files in: jersey-detection/app/model/")
print(f"2. Update .gitignore to whitelist new filenames")
print(f"3. Update roboflow_detector.py to load new models")
print(f"4. git add + commit + push → Railway auto-deploys")
print(f"{'=' * 60}")